LegalMateChatBot-qwen2.5 1.5billion parameters

fine tunning on legal data and urdu QAs

In [42]:
!pip install -q transformers accelerate datasets peft bitsandbytes safetensors pandas pyarrow
# -------- STEP 2: Mount Google Drive --------
from google.colab import drive
drive.mount('/content/drive')
import os
import json
import math
import datetime
from typing import Dict
import pandas as pd
from datasets import Dataset, concatenate_datasets
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
# -------- USER CONFIG --------
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "/content/drive/MyDrive/LegalMateDat/qwen2.5/fineTuned"
DATASET_PATHS = {
    "filtered_property": "/content/drive/MyDrive/LegalMateData/qwen2.5/trainingData/filtered_property_laws_qa.jsonl",
    "supreme_bi": "/content/drive/MyDrive/LegalMateData/qwen2.5/trainingData/legalUQA_supreme_bilingual.parquet",
    "legalUQA": "/content/drive/MyDrive/LegalMateData/qwen2.5/trainingData/legalUQA.parquet",
    "UQAD": "/content/drive/MyDrive/LegalMateData/qwen2.5/trainingData/UQAD.json",
}

# LoRA Config
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj", "fc1", "fc2", "wte"]

# Training hyperparams (optimized for 12GB GPU)
EPOCHS = 1
LEARNING_RATE = 2e-4
PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
MAX_SEQ_LENGTH = 512

# -------- Auto logging setup --------
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
log_dir = f"{OUTPUT_DIR}_logs_{timestamp}"
os.makedirs(log_dir, exist_ok=True)
print(f"🔹 Logs will be saved at: {log_dir}")

# -------- Dataset Loading Functions --------
def load_filtered_property(path: str) -> Dataset:
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            obj = json.loads(line)
            instr = obj.get('instruction', '').strip()
            inp = obj.get('input', '').strip()
            prompt = instr if instr else inp
            records.append({"instruction": prompt, "output": ""})
    return Dataset.from_list(records)


def load_supreme_bilingual(path: str) -> Dataset:
    df = pd.read_parquet(path)
    rows = [{"instruction": str(r['input']).strip(), "output": str(r['output']).strip()} for _, r in df.iterrows()]
    return Dataset.from_pandas(pd.DataFrame(rows))


def load_legalUQA_parquet(path: str) -> Dataset:
    df = pd.read_parquet(path)
    rows = []
    for _, r in df.iterrows():
        txt = str(r['text'])
        if '[INST]' in txt and '[/INST]' in txt:
            instr = txt.split('[INST]')[1].split('[/INST]')[0].strip()
            resp = txt.split('[/INST]')[-1].strip()
        else:
            instr, resp = txt, ''
        rows.append({"instruction": instr, "output": resp})
    return Dataset.from_pandas(pd.DataFrame(rows))


def load_UQAD_json(path: str) -> Dataset:
    with open(path, 'r', encoding='utf-8') as f:
        js = json.load(f)
    rows = []
    for topk, topv in js.items():
        context = topv.get('context', '').strip()
        questions = topv.get('question', {})
        for _, qv in questions.items():
            q = qv.get('question', '').strip()
            ans_list = [str(a).strip() for _, a in qv.get('answer', {}).items()]
            ans = ans_list[0] if ans_list else ''
            instr = f"Context: {context}\nQuestion: {q}"
            rows.append({"instruction": instr, "output": ans})
    return Dataset.from_pandas(pd.DataFrame(rows))


def build_combined_dataset(files: Dict[str, str]) -> Dataset:
    pieces = []
    loaders = {
        'filtered_property': load_filtered_property,
        'supreme_bi': load_supreme_bilingual,
        'legalUQA': load_legalUQA_parquet,
        'UQAD': load_UQAD_json,
    }
    for key, path in files.items():
        if os.path.exists(path):
            print(f"📂 Loading {key} dataset from: {path}")
            pieces.append(loaders[key](path))
        else:
            print(f"⚠️ Skipped missing file: {path}")
    if not pieces:
        raise FileNotFoundError("No dataset files found.")
    combined = concatenate_datasets(pieces)
    combined = combined.filter(lambda x: len(x['instruction'].strip()) > 0)
    return combined


def make_prompt(instruction, output=None):
    if output is None or not output.strip():
        return f"<s>[INST] {instruction} [/INST]"
    return f"<s>[INST] {instruction} [/INST] {output} </s>"


def tokenize_fn(examples, tokenizer):
    prompts = [make_prompt(i, o) for i, o in zip(examples['instruction'], examples['output'])]
    tokenized = tokenizer(prompts, truncation=True, max_length=MAX_SEQ_LENGTH, padding='max_length')
    tokenized['labels'] = tokenized['input_ids'].copy()
    return tokenized

# -------- Main Fine-tuning Logic --------
def main():
    print("📘 Preparing dataset...")
    ds = build_combined_dataset(DATASET_PATHS)
    print(f"✅ Combined dataset size: {len(ds)}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenized = ds.map(lambda ex: tokenize_fn(ex, tokenizer), batched=True, remove_columns=ds.column_names)

    print("⚙️ Loading model in 8-bit mode...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        load_in_8bit=True,
        device_map='auto',
        trust_remote_code=True,
    )
    model = prepare_model_for_kbit_training(model)

    print("🔧 Applying LoRA configuration...")
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=TARGET_MODULES,
        lora_dropout=LORA_DROPOUT,
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    def collate_fn(batch):
        return {
            'input_ids': torch.tensor([b['input_ids'] for b in batch]),
            'attention_mask': torch.tensor([b['attention_mask'] for b in batch]),
            'labels': torch.tensor([b['labels'] for b in batch]),
        }

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=True,
        logging_steps=50,
        save_strategy="epoch",
        save_total_limit=2,
        remove_unused_columns=False,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized,
        data_collator=collate_fn,
    )

    print("🚀 Starting fine-tuning...\n")
    trainer.train()

    print("💾 Saving LoRA adapter...")
    model.save_pretrained(OUTPUT_DIR)
        # -------- STEP 14: Log training summary --------
    with open(f"{log_dir}/summary.txt", "w") as f:
        f.write(f"Model: {MODEL_NAME}\n")
        f.write(f"Output dir: {OUTPUT_DIR}\n")
        f.write(f"Training date: {timestamp}\n")
        f.write(f"Epochs: {EPOCHS}\n")
        f.write(f"Learning rate: {LEARNING_RATE}\n")
        f.write(f"Batch size: {PER_DEVICE_BATCH_SIZE}\n")
        f.write(f"Gradient Accumulation: {GRAD_ACCUM_STEPS}\n")

    print(f"📝 Training summary logged in {log_dir}/summary.txt")
    print("✅ All done!")

if __name__ == '__main__':
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔹 Logs will be saved at: /content/drive/MyDrive/LegalMateDat/qwen2.5/fineTuned_logs_20251101_072116
📘 Preparing dataset...
📂 Loading filtered_property dataset from: /content/drive/MyDrive/LegalMateData/qwen2.5/trainingData/filtered_property_laws_qa.jsonl
📂 Loading supreme_bi dataset from: /content/drive/MyDrive/LegalMateData/qwen2.5/trainingData/legalUQA_supreme_bilingual.parquet
📂 Loading legalUQA dataset from: /content/drive/MyDrive/LegalMateData/qwen2.5/trainingData/legalUQA.parquet
📂 Loading UQAD dataset from: /content/drive/MyDrive/LegalMateData/qwen2.5/trainingData/UQAD.json


Filter:   0%|          | 0/6793 [00:00<?, ? examples/s]

✅ Combined dataset size: 6793


Map:   0%|          | 0/6793 [00:00<?, ? examples/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


⚙️ Loading model in 8-bit mode...
🔧 Applying LoRA configuration...
trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815
🚀 Starting fine-tuning...



/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss
50,1.696800
100,1.067600
150,0.982400
200,0.870100
250,0.951600
300,0.924800
350,0.904100
400,0.850000
450,0.908000
500,0.791300


💾 Saving LoRA adapter...
📝 Training summary logged in /content/drive/MyDrive/LegalMateDat/qwen2.5/fineTuned_logs_20251101_072116/summary.txt
✅ All done!
